# 01a. Tiền xử lý dữ liệu văn bản

Khảo sát phân bố dữ liệu, xây dựng quy trình chuẩn hóa chuỗi ký tự, teencode và tách từ tiếng Việt cho bộ dữ liệu ViMMSD phục vụ mô hình ngôn ngữ PhoBERT.

In [ ]:
# Thiết lập môi trường thực thi
import os, subprocess, sys
from pathlib import Path

REPO = "github.com/yunaLee21/Multimodal_Sarcasm_Detection.git"
REF = "main"
CODE_DIR = "/tmp/IE403"

def get_github_token():
    try:
        if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret("GITHUB_TOKEN")
        from google.colab import userdata
        return userdata.get("GITHUB_TOKEN")
    except Exception:
        return None

ON_CLOUD = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE")) or "COLAB_RELEASE_TAG" in os.environ
if ON_CLOUD:
    if "COLAB_RELEASE_TAG" in os.environ:
        from google.colab import drive
        drive.mount("/content/drive")
    token = get_github_token()
    url = f"https://{token}@{REPO}" if token else f"https://{REPO}"
    subprocess.run(["rm", "-rf", CODE_DIR])
    subprocess.run(["git", "clone", "-q", url, CODE_DIR], check=True)
    subprocess.run(["git", "-C", CODE_DIR, "checkout", "-q", REF], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{CODE_DIR}/requirements-kaggle.txt"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", CODE_DIR], check=True)
    if CODE_DIR + "/src" not in sys.path:
        sys.path.insert(0, CODE_DIR + "/src")
    os.chdir(CODE_DIR)
elif Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)

print("Thư mục làm việc:", os.getcwd())

In [ ]:
import json, re
from collections import Counter
import pandas as pd
import emoji

from vimmsd.data.preprocessing import clean_text, TextPreprocessor, TEENCODE
from vimmsd.utils.config import load_config

# Nạp cấu hình và dữ liệu tập huấn luyện
cfg = load_config("configs/base.yaml")
train_json_path = Path(cfg.paths.data_dir) / "vimmsd-train.json"

with open(train_json_path, encoding="utf-8") as f:
    raw_train = json.load(f)

df_train = pd.DataFrame(list(raw_train.values()))
print(f"Nạp thành công {len(df_train):,} mẫu dữ liệu từ {train_json_path.name}")
df_train.head()

## 1. Khảo sát đặc tính dữ liệu văn bản

Thống kê phân bố nhãn, độ dài văn bản và các dạng nhiễu thường gặp trên tập huấn luyện.

In [ ]:
# Phân bố nhãn bài toán
label_dist = df_train["label"].value_counts()
label_pct = df_train["label"].value_counts(normalize=True).mul(100)
summary_labels = pd.DataFrame({"Số lượng": label_dist, "Tỷ lệ (%)": label_pct.round(2)})
print("Bảng phân bố nhãn:")
display(summary_labels)

# Thống kê độ dài văn bản
df_train["word_count"] = df_train["caption"].apply(lambda c: len(str(c).split()))
df_train["char_count"] = df_train["caption"].apply(lambda c: len(str(c)))

print("Thống kê độ dài:")
display(df_train[["word_count", "char_count"]].describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]).round(1))

In [ ]:
# Thống kê các dạng nhiễu mạng xã hội
captions = df_train["caption"].fillna("").tolist()

url_cnt = sum(1 for c in captions if re.search(r"https?://|www\.", c))
mention_cnt = sum(1 for c in captions if re.search(r"@\w+", c))
hash_cnt = sum(1 for c in captions if "#" in c)
html_cnt = sum(1 for c in captions if re.search(r"&[a-zA-Z]+;|&#\d+;", c))
repeat_cnt = sum(1 for c in captions if re.search(r"([a-zA-ZÀ-ỹ])\1{2,}", c, re.IGNORECASE))
emoji_cnt = sum(1 for c in captions if emoji.emoji_count(c) > 0)

noise_summary = pd.DataFrame([
    {"Đặc trưng": "Hashtag", "Số mẫu xuất hiện": hash_cnt, "Tỷ lệ (%)": hash_cnt / len(captions) * 100},
    {"Đặc trưng": "Emoji", "Số mẫu xuất hiện": emoji_cnt, "Tỷ lệ (%)": emoji_cnt / len(captions) * 100},
    {"Đặc trưng": "Ký tự lặp", "Số mẫu xuất hiện": repeat_cnt, "Tỷ lệ (%)": repeat_cnt / len(captions) * 100},
    {"Đặc trưng": "Thực thể HTML", "Số mẫu xuất hiện": html_cnt, "Tỷ lệ (%)": html_cnt / len(captions) * 100},
    {"Đặc trưng": "Đường dẫn URL", "Số mẫu xuất hiện": url_cnt, "Tỷ lệ (%)": url_cnt / len(captions) * 100},
    {"Đặc trưng": "Đề cập người dùng", "Số mẫu xuất hiện": mention_cnt, "Tỷ lệ (%)": mention_cnt / len(captions) * 100},
])
display(noise_summary.round(2))

In [ ]:
# Thống kê tần suất emoji
all_emojis = []
for c in captions:
    all_emojis.extend(emoji.distinct_emoji_list(c))

top_emojis = Counter(all_emojis).most_common(12)
emoji_analysis = pd.DataFrame([
    {"Emoji": em, "Mô tả emoji": emoji.demojize(em), "Tần suất": cnt}
    for em, cnt in top_emojis
])
print("Top 12 emoji xuất hiện nhiều nhất:")
display(emoji_analysis)

## 2. Đánh giá quy trình tiền xử lý văn bản

Kiểm tra kết quả làm sạch, chuẩn hóa teencode và tách từ tiếng Việt trên các mẫu thực tế.

In [ ]:
# Đánh giá trên các mẫu đại diện
selected_indices = [0, 12, 42, 88, 105, 170, 310, 440, 852, 1029]
eval_rows = []

for idx in selected_indices:
    raw = captions[idx]
    cleaned_demojize = clean_text(raw, emoji_mode="demojize", word_segment_=True)
    cleaned_keep = clean_text(raw, emoji_mode="keep", word_segment_=True)
    eval_rows.append({
        "Index": idx,
        "Văn bản gốc": raw,
        "Tiền xử lý demojize và tách từ": cleaned_demojize,
        "Tiền xử lý giữ nguyên emoji": cleaned_keep,
    })

pd.set_option("display.max_colwidth", None)
display(pd.DataFrame(eval_rows))

## 3. Kiểm tra tính toàn vẹn ngữ nghĩa trên các trường hợp đặc biệt

Xác thực khả năng bảo toàn các từ viết tắt chuyên môn và từ vựng đặc thù.

In [ ]:
edge_cases = [
    ("Từ vựng đặc thù", "Ăn 2 kg sữa chua cay, sinh SN 1999, mua bảo hiểm Dr.G và biết PK game"),
    ("Teencode và ký tự lặp", "Phục vụ đỉnh chóp lun á, đợi có 2 tiếng thui à khum quạo tí nào ngonnnn 🐧"),
    ("Thực thể HTML, URL và mention", "Ad ơi cmt này đỉnh vch &amp; link https://fb.com #review @admin"),
    ("Dấu câu và emoji", "Đồ ăn ngon tuyệt vời... Ăn xong vô viện lun ak ??? !!! 🤡"),
]

for category, text in edge_cases:
    print(f"[{category}]")
    print("  Gốc      :", text)
    print("  Chuẩn hóa:", clean_text(text, emoji_mode="demojize", word_segment_=True))
    print()

## 4. Đo lường hiệu năng và cơ chế lưu trữ đệm

Đánh giá tốc độ xử lý của TextPreprocessor khi áp dụng cơ chế lưu cache ra đĩa.

In [ ]:
import time

preprocessor = TextPreprocessor.from_config(cfg.data.text)
benchmark_sample = captions[:1000]

# Thực thi lần đầu và tạo cache
t0 = time.time()
res_cold = preprocessor.process_all(benchmark_sample, cache_dir="data/processed")
t_cold = time.time() - t0

# Thực thi lần hai đọc từ cache
t1 = time.time()
res_warm = preprocessor.process_all(benchmark_sample, cache_dir="data/processed")
t_warm = time.time() - t1

print(f"Thời gian xử lý lần đầu: {t_cold:.3f} giây")
print(f"Thời gian nạp từ cache: {t_warm:.4f} giây")
print(f"Hệ số tăng tốc: {t_cold / max(t_warm, 1e-6):.1f}x")
assert res_cold == res_warm, "Dữ liệu cache không khớp"
print("Kiểm tra cache: Hợp lệ")

## 5. Kết luận và cấu hình thực nghiệm

1. Chuẩn hóa chuỗi ký tự: Quy trình xử lý triệt để các dạng nhiễu mạng xã hội, teencode thông dụng và bảo toàn chính xác các từ vựng đặc thù.
2. Tách từ tiếng Việt: Định dạng nối từ ghép bằng dấu gạch dưới tương thích với bộ từ vựng của mô hình PhoBERT.
3. Cấu hình độ dài chuỗi: Phân vị độ dài p95 đạt 230 từ. Cấu hình max_length = 256 đảm bảo bao phủ hơn 96% độ dài mẫu dữ liệu.
4. Xử lý biểu tượng cảm xúc: Sử dụng chế độ demojize để chuyển đổi biểu tượng cảm xúc thành văn bản ngữ nghĩa.